### Load the Bronze tables

In [0]:
from pyspark.sql import functions as F

customers_bronze = spark.table("workspace.commerce_dataset.bronze_customers")
orders_bronze = spark.table("workspace.commerce_dataset.bronze_orders")
order_items_bronze = spark.table("workspace.commerce_dataset.bronze_order_items")
products_bronze = spark.table("workspace.commerce_dataset.bronze_products")
payments_bronze = spark.table("workspace.commerce_dataset.bronze_payments")

print(f"Customers: {customers_bronze.count():,}")
print(f"Orders: {orders_bronze.count():,}")
print(f"Order items: {order_items_bronze.count():,}")
print(f"Products: {products_bronze.count():,}")
print(f"Payments: {payments_bronze.count():,}")


### Create the city standardization rule

In [0]:
def standardize_city(column):
    city = F.lower(F.trim(column))

    return (
        F.when(city.isNull() | (city == ""), "Unknown")
        .when(city.isin("prishtina", "prishtinë"), "Prishtinë")
        .when(city.isin("gjakove", "gjakovë"), "Gjakovë")
        .when(city.isin("peja", "pejë"), "Pejë")
        .when(city.isin("mitrovice", "mitrovicë"), "Mitrovicë")
        .when(city.isin("vushtrri", "vushtrria"), "Vushtrri")
        .when(city == "prizren", "Prizren")
        .when(city == "gjilan", "Gjilan")
        .when(city == "ferizaj", "Ferizaj")
        .otherwise(F.initcap(F.trim(column)))
    )


### Remove and quarantine duplicate customers

In [0]:
exact_customer_duplicates = customers_bronze.groupBy(customers_bronze.columns).count().filter(F.col("count") > 1).drop("count")
customers_duplicates = customers_bronze.join(exact_customer_duplicates, on=customers_bronze.columns, how="inner").dropDuplicates().withColumn("validation_reason", F.lit("exact_duplicate"))

customers_valid = customers_bronze.dropDuplicates()

duplicate_customer_ids = customers_valid.groupBy("customer_id").count().filter(F.col("count") > 1).select("customer_id")
customers_conflicts = customers_valid.join(duplicate_customer_ids, on="customer_id", how="inner").withColumn("validation_reason", F.lit("conflicting_customer_id"))
customers_valid = customers_valid.join(duplicate_customer_ids, on="customer_id", how="left_anti")

print(f"Duplicate customer copies quarantined: {customers_duplicates.count():,}")
print(f"Conflicting customer IDs: {duplicate_customer_ids.count():,}")


### Validate and clean customers

In [0]:
customers_checked = customers_valid.withColumn(
    "validation_reason",
    F.when(F.col("customer_id").isNull(), "missing_customer_id")
    .when(F.col("customer_id") <= 0, "invalid_customer_id")
)

customers_invalid = customers_checked.filter(F.col("validation_reason").isNotNull())
customers_valid = customers_checked.filter(F.col("validation_reason").isNull()).drop("validation_reason")

customers_quarantine = (
    customers_duplicates.select("customer_id", "customer_name", "email", "city", "customer_type", "_source_file", "validation_reason")
    .union(customers_conflicts.select("customer_id", "customer_name", "email", "city", "customer_type", "_source_file", "validation_reason"))
    .union(customers_invalid.select("customer_id", "customer_name", "email", "city", "customer_type", "_source_file", "validation_reason"))
)

customers_silver = (
    customers_valid
    .withColumn("customer_name", F.when(F.col("customer_name").isNull() | (F.trim(F.col("customer_name")) == ""), "Unknown").otherwise(F.trim(F.col("customer_name"))))
    .withColumn("email", F.when(F.col("email").isNull() | ~F.trim(F.col("email")).rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$"), F.lit(None)).otherwise(F.lower(F.trim(F.col("email")))))
    .withColumn("city", standardize_city(F.col("city")))
    .withColumn("_customer_type", F.lower(F.trim(F.col("customer_type"))))
    .withColumn("customer_type", F.when(F.col("_customer_type") == "retail", "Retail").when(F.col("_customer_type") == "business", "Business").when(F.col("_customer_type") == "vip", "VIP").otherwise("Unknown"))
    .drop("_customer_type")
)

print(f"Trusted customers: {customers_silver.count():,}")
print(f"Quarantined customers: {customers_quarantine.count():,}")
display(customers_silver.limit(10))


### Remove and quarantine duplicate products

In [0]:
exact_product_duplicates = products_bronze.groupBy(products_bronze.columns).count().filter(F.col("count") > 1).drop("count")
products_duplicates = products_bronze.join(exact_product_duplicates, on=products_bronze.columns, how="inner").dropDuplicates().withColumn("validation_reason", F.lit("exact_duplicate"))

products_valid = products_bronze.dropDuplicates()

duplicate_product_ids = products_valid.groupBy("product_id").count().filter(F.col("count") > 1).select("product_id")
products_conflicts = products_valid.join(duplicate_product_ids, on="product_id", how="inner").withColumn("validation_reason", F.lit("conflicting_product_id"))
products_valid = products_valid.join(duplicate_product_ids, on="product_id", how="left_anti")

print(f"Duplicate product copies quarantined: {products_duplicates.count():,}")
print(f"Conflicting product IDs: {duplicate_product_ids.count():,}")


### Validate and clean products

In [0]:
products_checked = products_valid.withColumn(
    "validation_reason",
    F.when(F.col("product_id").isNull(), "missing_product_id")
    .when(F.col("product_id") <= 0, "invalid_product_id")
    .when(F.col("product_name").isNull() | (F.trim(F.col("product_name")) == ""), "missing_product_name")
    .when(F.col("list_price").isNull() | (F.col("list_price") <= 0), "invalid_list_price")
)

products_invalid = products_checked.filter(F.col("validation_reason").isNotNull())
products_valid = products_checked.filter(F.col("validation_reason").isNull()).drop("validation_reason")

products_quarantine = (
    products_duplicates.select("product_id", "product_name", "category", "list_price", "_source_file", "validation_reason")
    .union(products_conflicts.select("product_id", "product_name", "category", "list_price", "_source_file", "validation_reason"))
    .union(products_invalid.select("product_id", "product_name", "category", "list_price", "_source_file", "validation_reason"))
)

products_silver = (
    products_valid
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("category", F.when(F.col("category").isNull() | (F.trim(F.col("category")) == ""), "Unknown").otherwise(F.initcap(F.trim(F.col("category")))))
)

print(f"Trusted products: {products_silver.count():,}")
print(f"Quarantined products: {products_quarantine.count():,}")
display(products_silver.limit(10))


### Remove and quarantine duplicate orders

In [0]:
exact_order_duplicates = orders_bronze.groupBy(orders_bronze.columns).count().filter(F.col("count") > 1).drop("count")
orders_duplicates = orders_bronze.join(exact_order_duplicates, on=orders_bronze.columns, how="inner").dropDuplicates().withColumn("validation_reason", F.lit("exact_duplicate"))

orders_valid = orders_bronze.dropDuplicates()

duplicate_order_ids = orders_valid.groupBy("order_id").count().filter(F.col("count") > 1).select("order_id")
orders_conflicts = orders_valid.join(duplicate_order_ids, on="order_id", how="inner").withColumn("validation_reason", F.lit("conflicting_order_id"))
orders_valid = orders_valid.join(duplicate_order_ids, on="order_id", how="left_anti")

print(f"Duplicate order copies quarantined: {orders_duplicates.count():,}")
print(f"Conflicting order IDs: {duplicate_order_ids.count():,}")


### Validate orders

In [0]:
orders_working = (
    orders_valid
    .withColumn("_order_date_parsed", F.expr("try_cast(order_date as date)"))
    .withColumn("_order_status_normalized", F.lower(F.trim(F.col("order_status"))))
    .withColumn("_order_status_normalized", F.when(F.col("_order_status_normalized") == "canceled", "cancelled").otherwise(F.col("_order_status_normalized")))
)

orders_checked = orders_working.withColumn(
    "validation_reason",
    F.when(F.col("order_id").isNull(), "missing_order_id")
    .when(F.col("order_id") <= 0, "invalid_order_id")
    .when(F.col("customer_id").isNull(), "missing_customer_id")
    .when(F.col("customer_id") <= 0, "invalid_customer_id")
    .when(F.col("_order_date_parsed").isNull(), "invalid_order_date")
    .when(F.col("_order_status_normalized").isNull() | ~F.col("_order_status_normalized").isin("completed", "pending", "cancelled", "refunded"), "unexpected_order_status")
)

orders_invalid = orders_checked.filter(F.col("validation_reason").isNotNull())
orders_valid = orders_checked.filter(F.col("validation_reason").isNull()).drop("validation_reason")

orders_without_customer = orders_valid.join(customers_silver.select("customer_id"), on="customer_id", how="left_anti").withColumn("validation_reason", F.lit("customer_not_in_trusted_customers"))
orders_valid = orders_valid.join(customers_silver.select("customer_id"), on="customer_id", how="inner")

orders_quarantine = (
    orders_duplicates.select("order_id", "customer_id", "order_date", "shipping_city", "order_status", "_source_file", "validation_reason")
    .union(orders_conflicts.select("order_id", "customer_id", "order_date", "shipping_city", "order_status", "_source_file", "validation_reason"))
    .union(orders_invalid.select("order_id", "customer_id", "order_date", "shipping_city", "order_status", "_source_file", "validation_reason"))
    .union(orders_without_customer.select("order_id", "customer_id", "order_date", "shipping_city", "order_status", "_source_file", "validation_reason"))
)

orders_silver = (
    orders_valid
    .withColumn("order_date", F.col("_order_date_parsed"))
    .withColumn("shipping_city", standardize_city(F.col("shipping_city")))
    .withColumn("order_status", F.col("_order_status_normalized"))
    .drop("_order_date_parsed", "_order_status_normalized")
    .select("order_id", "customer_id", "order_date", "shipping_city", "order_status", "_source_file")
)

print(f"Trusted orders: {orders_silver.count():,}")
print(f"Quarantined orders: {orders_quarantine.count():,}")
display(orders_silver.limit(10))


### Remove and quarantine duplicate order items

In [0]:
exact_order_item_duplicates = order_items_bronze.groupBy(order_items_bronze.columns).count().filter(F.col("count") > 1).drop("count")
order_items_duplicates = order_items_bronze.join(exact_order_item_duplicates, on=order_items_bronze.columns, how="inner").dropDuplicates().withColumn("validation_reason", F.lit("exact_duplicate"))

order_items_valid = order_items_bronze.dropDuplicates()

duplicate_order_item_ids = order_items_valid.groupBy("order_item_id").count().filter(F.col("count") > 1).select("order_item_id")
order_items_conflicts = order_items_valid.join(duplicate_order_item_ids, on="order_item_id", how="inner").withColumn("validation_reason", F.lit("conflicting_order_item_id"))
order_items_valid = order_items_valid.join(duplicate_order_item_ids, on="order_item_id", how="left_anti")

print(f"Duplicate order-item copies quarantined: {order_items_duplicates.count():,}")
print(f"Conflicting order-item IDs: {duplicate_order_item_ids.count():,}")


### Validate order items and relationships

In [0]:
order_items_checked = order_items_valid.withColumn(
    "validation_reason",
    F.when(F.col("order_item_id").isNull(), "missing_order_item_id")
    .when(F.col("order_item_id") <= 0, "invalid_order_item_id")
    .when(F.col("order_id").isNull(), "missing_order_id")
    .when(F.col("order_id") <= 0, "invalid_order_id")
    .when(F.col("product_id").isNull(), "missing_product_id")
    .when(F.col("product_id") <= 0, "invalid_product_id")
    .when(F.col("quantity").isNull() | (F.col("quantity") <= 0), "invalid_quantity")
    .when(F.col("unit_price").isNull() | (F.col("unit_price") <= 0), "invalid_unit_price")
)

order_items_invalid = order_items_checked.filter(F.col("validation_reason").isNotNull())
order_items_valid = order_items_checked.filter(F.col("validation_reason").isNull()).drop("validation_reason")

items_without_order = order_items_valid.join(orders_silver.select("order_id"), on="order_id", how="left_anti").withColumn("validation_reason", F.lit("order_not_in_trusted_orders"))
order_items_valid = order_items_valid.join(orders_silver.select("order_id"), on="order_id", how="inner")

items_without_product = order_items_valid.join(products_silver.select("product_id"), on="product_id", how="left_anti").withColumn("validation_reason", F.lit("product_not_in_trusted_products"))
order_items_valid = order_items_valid.join(products_silver.select("product_id"), on="product_id", how="inner")

order_items_quarantine = (
    order_items_duplicates.select("order_item_id", "order_id", "product_id", "quantity", "unit_price", "_source_file", "validation_reason")
    .union(order_items_conflicts.select("order_item_id", "order_id", "product_id", "quantity", "unit_price", "_source_file", "validation_reason"))
    .union(order_items_invalid.select("order_item_id", "order_id", "product_id", "quantity", "unit_price", "_source_file", "validation_reason"))
    .union(items_without_order.select("order_item_id", "order_id", "product_id", "quantity", "unit_price", "_source_file", "validation_reason"))
    .union(items_without_product.select("order_item_id", "order_id", "product_id", "quantity", "unit_price", "_source_file", "validation_reason"))
)

order_items_silver = (
    order_items_valid
    .withColumn("line_total", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .select("order_item_id", "order_id", "product_id", "quantity", "unit_price", "line_total", "_source_file")
)

print(f"Trusted order items: {order_items_silver.count():,}")
print(f"Quarantined order items: {order_items_quarantine.count():,}")
display(order_items_silver.limit(10))


### Remove and quarantine duplicate payments

In [0]:
exact_payment_duplicates = payments_bronze.groupBy(payments_bronze.columns).count().filter(F.col("count") > 1).drop("count")
payments_duplicates = payments_bronze.join(exact_payment_duplicates, on=payments_bronze.columns, how="inner").dropDuplicates().withColumn("validation_reason", F.lit("exact_duplicate"))

payments_valid = payments_bronze.dropDuplicates()

duplicate_payment_ids = payments_valid.groupBy("payment_id").count().filter(F.col("count") > 1).select("payment_id")
payments_conflicts = payments_valid.join(duplicate_payment_ids, on="payment_id", how="inner").withColumn("validation_reason", F.lit("conflicting_payment_id"))
payments_valid = payments_valid.join(duplicate_payment_ids, on="payment_id", how="left_anti")

print(f"Duplicate payment copies quarantined: {payments_duplicates.count():,}")
print(f"Conflicting payment IDs: {duplicate_payment_ids.count():,}")


### Validate payments

In [0]:
payments_working = (
    payments_valid
    .withColumn("_payment_date_parsed", F.expr("try_cast(payment_date as date)"))
    .withColumn("_payment_status_normalized", F.lower(F.trim(F.col("payment_status"))))
    .withColumn("_payment_method_normalized", F.lower(F.trim(F.col("payment_method"))))
    .withColumn("_payment_method_normalized", F.when(F.col("_payment_method_normalized") == "bank_transfer", "bank transfer").otherwise(F.col("_payment_method_normalized")))
)

invalid_payment_amount = (
    F.col("amount").isNull() |
    ((F.col("_payment_status_normalized") == "paid") & (F.col("amount") <= 0)) |
    ((F.col("_payment_status_normalized") == "failed") & (F.col("amount") != 0)) |
    ((F.col("_payment_status_normalized") == "refunded") & (F.col("amount") >= 0))
)

payments_checked = payments_working.withColumn(
    "validation_reason",
    F.when(F.col("payment_id").isNull(), "missing_payment_id")
    .when(F.col("payment_id") <= 0, "invalid_payment_id")
    .when(F.col("order_id").isNull(), "missing_order_id")
    .when(F.col("order_id") <= 0, "invalid_order_id")
    .when(F.col("_payment_date_parsed").isNull(), "invalid_payment_date")
    .when(F.col("_payment_status_normalized").isNull() | ~F.col("_payment_status_normalized").isin("paid", "failed", "refunded"), "unexpected_payment_status")
    .when(F.col("_payment_method_normalized").isNull() | ~F.col("_payment_method_normalized").isin("card", "bank transfer", "paypal", "cash"), "unexpected_payment_method")
    .when(invalid_payment_amount, "amount_inconsistent_with_payment_status")
)

payments_invalid = payments_checked.filter(F.col("validation_reason").isNotNull())
payments_valid = payments_checked.filter(F.col("validation_reason").isNull()).drop("validation_reason")

payments_without_order = payments_valid.join(orders_silver.select("order_id"), on="order_id", how="left_anti").withColumn("validation_reason", F.lit("order_not_in_trusted_orders"))
payments_valid = payments_valid.join(orders_silver.select("order_id", F.col("order_date").alias("_trusted_order_date")), on="order_id", how="inner")

payments_before_order = payments_valid.filter(F.col("_payment_date_parsed") < F.col("_trusted_order_date")).withColumn("validation_reason", F.lit("payment_date_before_order_date"))
payments_valid = payments_valid.filter(F.col("_payment_date_parsed") >= F.col("_trusted_order_date"))

payments_quarantine = (
    payments_duplicates.select("payment_id", "order_id", "amount", "payment_method", "payment_status", "payment_date", "_source_file", "validation_reason")
    .union(payments_conflicts.select("payment_id", "order_id", "amount", "payment_method", "payment_status", "payment_date", "_source_file", "validation_reason"))
    .union(payments_invalid.select("payment_id", "order_id", "amount", "payment_method", "payment_status", "payment_date", "_source_file", "validation_reason"))
    .union(payments_without_order.select("payment_id", "order_id", "amount", "payment_method", "payment_status", "payment_date", "_source_file", "validation_reason"))
    .union(payments_before_order.select("payment_id", "order_id", "amount", "payment_method", "payment_status", "payment_date", "_source_file", "validation_reason"))
)

payments_silver = (
    payments_valid
    .withColumn("payment_date", F.col("_payment_date_parsed"))
    .withColumn("payment_status", F.col("_payment_status_normalized"))
    .withColumn("payment_method", F.when(F.col("_payment_method_normalized") == "card", "Card").when(F.col("_payment_method_normalized") == "bank transfer", "Bank Transfer").when(F.col("_payment_method_normalized") == "paypal", "PayPal").when(F.col("_payment_method_normalized") == "cash", "Cash"))
    .drop("_payment_date_parsed", "_payment_status_normalized", "_payment_method_normalized", "_trusted_order_date")
    .select("payment_id", "order_id", "amount", "payment_method", "payment_status", "payment_date", "_source_file")
)

print(f"Trusted payments: {payments_silver.count():,}")
print(f"Quarantined payments: {payments_quarantine.count():,}")
display(payments_silver.limit(10))


### Check the final Silver relationships

In [0]:
orders_without_trusted_customer = orders_silver.join(customers_silver.select("customer_id"), on="customer_id", how="left_anti").count()
items_without_trusted_order = order_items_silver.join(orders_silver.select("order_id"), on="order_id", how="left_anti").count()
items_without_trusted_product = order_items_silver.join(products_silver.select("product_id"), on="product_id", how="left_anti").count()
payments_without_trusted_order = payments_silver.join(orders_silver.select("order_id"), on="order_id", how="left_anti").count()

print(f"Orders without trusted customer: {orders_without_trusted_customer}")
print(f"Order items without trusted order: {items_without_trusted_order}")
print(f"Order items without trusted product: {items_without_trusted_product}")
print(f"Payments without trusted order: {payments_without_trusted_order}")
print(f"Order items with non-positive quantity: {order_items_silver.filter(F.col('quantity') <= 0).count()}")
print(f"Order items with non-positive unit price: {order_items_silver.filter(F.col('unit_price') <= 0).count()}")


### Save the Silver and quarantine tables

In [0]:
customers_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.silver_customers")
products_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.silver_products")
orders_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.silver_orders")
order_items_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.silver_order_items")
payments_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.silver_payments")

customers_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.quarantine_customers")
products_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.quarantine_products")
orders_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.quarantine_orders")
order_items_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.quarantine_order_items")
payments_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.commerce_dataset.quarantine_payments")

print("Silver and quarantine Delta tables saved successfully.")


### Show the Silver summary

In [0]:
print("SILVER SUMMARY")
print(f"Customers: {customers_silver.count():,} trusted / {customers_bronze.count():,} raw / {customers_quarantine.count():,} quarantined")
print(f"Products: {products_silver.count():,} trusted / {products_bronze.count():,} raw / {products_quarantine.count():,} quarantined")
print(f"Orders: {orders_silver.count():,} trusted / {orders_bronze.count():,} raw / {orders_quarantine.count():,} quarantined")
print(f"Order items: {order_items_silver.count():,} trusted / {order_items_bronze.count():,} raw / {order_items_quarantine.count():,} quarantined")
print(f"Payments: {payments_silver.count():,} trusted / {payments_bronze.count():,} raw / {payments_quarantine.count():,} quarantined")
